# Nested cross-validation model comparison

The study has 299 observations, so scores from one split are unstable. This notebook compares 14 probability-capable classifier families with and without feature selection using the same 5-fold stratified outer splits. Each outer training partition has its own 5-fold stratified inner search. The inner search maximizes average precision for `DEATH_EVENT = 1`. Scaling, mutual-information feature selection, and parameter tuning are all contained within that inner search.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedKFold

from pro_gradu.data import load_dataset
from pro_gradu.evaluation import default_model_specs, evaluate_nested_cv

SEED = 42
df = load_dataset(Path('heart_failure_clinical_records_dataset.csv'))
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
specs = default_model_specs(include_feature_selection=True, random_state=SEED)
print(f'Evaluating {len(specs)} configurations across five paired outer folds.')

In [ ]:
fold_metrics, oof_predictions = evaluate_nested_cv(
    df, specs, outer_cv=outer_cv, inner_cv=inner_cv,
    scoring='average_precision', random_state=SEED,
)
fold_metrics.head()

## Fold-level results

The positive class is the observed death event (`DEATH_EVENT = 1`). ROC AUC and average precision use predicted class-1 probabilities. Sensitivity, specificity, balanced accuracy, and accuracy use the fixed threshold 0.5. Threshold selection is not tuned against held-out outer folds.

In [ ]:
metric_columns = ['roc_auc', 'average_precision', 'balanced_accuracy', 'sensitivity', 'specificity', 'accuracy']
summary = (fold_metrics.groupby(['model', 'feature_selection'])[metric_columns]
           .agg(['mean', 'std']).round(3)
           .sort_values(('average_precision', 'mean'), ascending=False))
summary

In [ ]:
# Keep fold-level results for audit/replotting. These generated files are ignored by Git.
output_dir = Path('results')
output_dir.mkdir(exist_ok=True)
fold_metrics.to_csv(output_dir / 'nested_cv_fold_metrics.csv', index=False)
oof_predictions.to_csv(output_dir / 'nested_cv_oof_predictions.csv')
print(f'Saved fold metrics and held-out predictions under {output_dir.resolve()}')

## Selection behavior

Each selected-feature model chooses `k` only inside its inner search. The selected names below are summarized across outer folds; different folds may retain different variables. They are not a definitive clinical ranking.

In [ ]:
selected = (fold_metrics[fold_metrics.feature_selection]
            .explode('selected_features')
            .groupby(['model', 'selected_features'], as_index=False)
            .size().rename(columns={'selected_features': 'feature', 'size': 'outer_folds_selected'}))
selected.sort_values(['model', 'outer_folds_selected'], ascending=[True, False])

## Limitations

Nested cross-validation reduces leakage from preprocessing and hyperparameter selection, but it is internal validation on a small cohort. Comparing many model families can still make the best observed mean score optimistic. Fold standard deviations describe split variability, not a formal confidence interval. These results are not external validation and are not a clinical tool. To claim a final winner, an independent cohort and a prespecified decision rule would be needed. The data contain follow-up time and right censoring; this binary task predicts an event observed during follow-up and does not estimate survival time. See `bayesian_survival_analysis.ipynb` for a separate exploratory time-to-event model.